In [2]:
import sys
!"{sys.executable}" -m pip install yfinance pandas numpy matplotlib sqlalchemy pyodbc

In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import os

In [4]:
assets = {
    "SPY":["SPDR S&P 500 ETF","EQUITY"],
    "QQQ":["Invesco QQQ ETF","Growth Equity"],
    "TLT":["ishares 20+ Year Treasury Bond ETF","Fixed Income"],
    "GLD":["SPDR Gold Shares","Gold"],
    "USO":["United States Oil Fund","Crude Oil"],
    "DBC":["Invesco DB Commodity Index Tracking Fund","Commodities"]
}

tickers = list(assets.keys())

start_date = "2015-01-01"
end_date = "2026-05-01"

In [5]:
raw = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=True
)

prices = raw["Close"].dropna()

prices.head()

C:\Users\anshb\anaconda3\Lib\site-packages\yfinance\scrapers\history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[*********************100%***********************]  6 of 6 completed


Ticker,DBC,GLD,QQQ,SPY,TLT,USO
Date,,,,,,
2015-01-02,15.412328,114.080002,94.665092,170.125000,93.367340,159.119995
2015-01-05,15.192514,115.800003,93.276436,167.052612,94.834000,150.320007
2015-01-06,15.048789,117.120003,92.025772,165.479172,96.542603,144.399994
2015-01-07,14.955792,116.430000,93.212097,167.541260,96.351898,146.960007
2015-01-08,15.014972,115.940002,94.996132,170.514206,95.075951,148.399994


In [6]:
price_table = prices.reset_index().melt(
    id_vars="Date",
    var_name="ticker",
    value_name="close_price",
)

price_table = price_table.rename(columns={"Date": "price_date"})

price_table.head()

,price_date,ticker,close_price
0,2015-01-02,DBC,15.412328
1,2015-01-05,DBC,15.192514
2,2015-01-06,DBC,15.048789
3,2015-01-07,DBC,14.955792
4,2015-01-08,DBC,15.014972


In [7]:
returns = prices.pct_change().dropna()

return_table = returns.reset_index().melt(
    id_vars="Date",
    var_name="ticker",
    value_name="daily_return"
)

return_table = return_table.rename(columns={"Date":"return_date"})

return_table.head()

,return_date,ticker,daily_return
0,2015-01-05,DBC,-0.014262
1,2015-01-06,DBC,-0.009460
2,2015-01-07,DBC,-0.006180
3,2015-01-08,DBC,0.003957
4,2015-01-09,DBC,-0.001689


In [8]:
asset_rows = []

for ticker, info in assets.items():
    asset_rows.append({
        "ticker":ticker,
        "asset_name":info[0],
        "asset_class": info[1]
    })

    asset_table = pd.DataFrame(asset_rows)

    asset_table

In [9]:
annual_return = returns.mean() * 252
annual_volatility = returns.std() * np.sqrt(252)
sharpe_ratio = annual_return / annual_volatility

cummulative = (1 + returns).cumprod()
drawdown = cummulative / cummulative.cummax() - 1
max_drawdown = drawdown.min()

risk_table = pd.DataFrame({
    "ticker": tickers,
    "annual_return":annual_return[tickers].values,
    "annual_volatility":annual_volatility[tickers].values,
    "sharpe_ratio":sharpe_ratio[tickers].values,
    "max_drawdown":max_drawdown[tickers].values
})

risk_table

,ticker,annual_return,annual_volatility,sharpe_ratio,max_drawdown
0,SPY,0.143263,0.176959,0.809584,-0.337173
1,QQQ,0.196959,0.218839,0.900020,-0.351187
2,TLT,0.003116,0.149108,0.020899,-0.483511
3,GLD,0.128776,0.158619,0.811855,-0.220022
4,USO,0.072878,0.396229,0.183930,-0.897695
5,DBC,0.078151,0.178548,0.437706,-0.417111


In [13]:
events = [
    ["2020-03-16", "COVID Emergency Fed cut", "Fed"],
    ["2022-03-16", "First Fed Rate Hike Cycle", "Fed"],
    ["2023-03-10","Silicon Valley Bank Collapse","Banking Stress"],
    ["2024-09-18","Fed Policy Decision","Fed"],
    ["2026-03-02","Geopolitical Risk Period","Geopolitical"]
]

event_table = pd.DataFrame(events)
event_table.columns=["event_date","event_name","event_type"]
event_table["event_date"] = pd.to_datetime(event_table["event_date"])

print(event_table.columns)
event_table

Index(['event_date', 'event_name', 'event_type'], dtype='str')


,event_date,event_name,event_type
0,2020-03-16,COVID Emergency Fed cut,Fed
1,2022-03-16,First Fed Rate Hike Cycle,Fed
2,2023-03-10,Silicon Valley Bank Collapse,Banking Stress
3,2024-09-18,Fed Policy Decision,Fed
4,2026-03-02,Geopolitical Risk Period,Geopolitical


In [15]:
server = "localhost"
database = "FinancialMarketAnalytics"

connection_string = (
    "mssql+pyodbc://@"
    + server +
    "/" + database +
    "?driver=ODBC+Driver17+for+SQL+Server"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)

In [17]:
import pyodbc
pyodbc.drivers()

['SQL Server',
 'Microsoft Access Driver (*.mdb, *.accdb)',
 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)',
 'Microsoft Access Text Driver (*.txt, *.csv)',
 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)',
 'ODBC Driver 17 for SQL Server',
 'ODBC Driver 18 for SQL Server']

In [18]:
from sqlalchemy import create_engine
import pandas as pd

server = "localhost"
database = "FinancialMarketAnalytics"
driver = "ODBC Driver 18 for SQL Server"

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    f"?driver={driver.replace(' ', '+')}"
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes"
)

engine = create_engine(connection_string)

test = pd.read_sql("SELECT name FROM sys.tables", engine)
test

C:\Users\anshb\anaconda3\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


,name
0,dim_asset
1,fact_prices
2,fact_returns
3,fact_risk_metrics
4,event_calendar


In [20]:
asset_table.to_sql("dim-asset",engine,if_exists="append",index=False)
price_table.to_sql("fact_prices",engine,if_exists="append",index=False)
return_table.to_sql("fact_returns",engine,if_exists="append",index=False)
risk_table.to_sql("fact_risk_metrics",engine,if_exists="append",index=False)
event_table.to_sql("event_calendar",engine,if_exists="append",index=False)


5

In [21]:
asset_table.to_sql("dim_asset", engine, if_exists="append", index=False)

print("Asset table loaded.")

Asset table loaded.


In [22]:

cumulative = (1 + returns).cumprod()


running_peak = cumulative.cummax()


drawdown = cumulative / running_peak - 1


drawdown_table = drawdown.reset_index().melt(
    id_vars="Date",
    var_name="ticker",
    value_name="drawdown"
)


drawdown_table = drawdown_table.rename(columns={"Date": "drawdown_date"})


drawdown_table.head()

,drawdown_date,ticker,drawdown
0,2015-01-05,DBC,0.000000
1,2015-01-06,DBC,-0.009460
2,2015-01-07,DBC,-0.015581
3,2015-01-08,DBC,-0.011686
4,2015-01-09,DBC,-0.013356


In [23]:
drawdown_table

,drawdown_date,ticker,drawdown
0,2015-01-05,DBC,0.000000
1,2015-01-06,DBC,-0.009460
2,2015-01-07,DBC,-0.015581
3,2015-01-08,DBC,-0.011686
4,2015-01-09,DBC,-0.013356
...,...,...,...
17077,2026-04-24,USO,-0.205091
17078,2026-04-27,USO,-0.191162
17079,2026-04-28,USO,-0.161864
17080,2026-04-29,USO,-0.095641


In [24]:
drawdown_table.to_sql(
    "fact_drawdowns",
    engine,
    if_exists="replace",
    index=False
)

print("fact_drawdowns table loaded successfully.")

fact_drawdowns table loaded successfully.
